# M&Ms dataset audit

Checks whether the Kaggle copy of M&Ms can support the study. Run the cells in order.

The study needs three things from the data:
1. A metadata file that gives the **vendor** of every patient.
2. **Labels** for patients from all four vendors (A, B, C, D).
3. Images in a form that keeps the **pixel spacing**, so scans can be resampled to a common resolution.

Nothing here trains a model or changes the dataset.

In [ ]:
# 1. Download the Kaggle copy (Colab may ask you to log in to Kaggle)
!pip -q install kagglehub nibabel
import kagglehub
root = kagglehub.dataset_download("tailength/m-and-ms-dataset")
print("Dataset folder:", root)

In [ ]:
# 2. Inventory: what kinds of files are in the dataset, and where
import os, collections

def ext_of(name):
    n = name.lower()
    return ".nii.gz" if n.endswith(".nii.gz") else os.path.splitext(n)[1] or "(none)"

files = []
for d, _, fs in os.walk(root):
    for f in fs:
        files.append(os.path.join(d, f))

by_ext = collections.Counter(ext_of(f) for f in files)
size_gb = sum(os.path.getsize(f) for f in files) / 1e9
print(f"{len(files)} files, {size_gb:.2f} GB")
for e, n in by_ext.most_common():
    print(f"  {e:10s} {n}")

print("\nFolders (3 levels deep):")
base_depth = root.rstrip(os.sep).count(os.sep)
for d, subdirs, fs in os.walk(root):
    depth = d.rstrip(os.sep).count(os.sep) - base_depth
    if depth <= 2:
        print("  " * depth + (os.path.basename(d) or d) + f"/   ({len(subdirs)} folders, {len(fs)} files)")
    subdirs.sort()

In [ ]:
# 3. Metadata: is there a table that gives the vendor of each patient?
import pandas as pd

tables = [f for f in files if f.lower().endswith((".csv", ".xlsx", ".xls", ".tsv"))]
meta, vendor_col, code_col = None, None, None
if not tables:
    print("NO metadata table found. Without vendor information the study cannot be run on this copy.")
for t in tables:
    try:
        if t.lower().endswith((".xlsx", ".xls")):
            df = pd.read_excel(t)
        else:
            df = pd.read_csv(t, sep=None, engine="python")
    except Exception as e:
        print("Could not read", t, "->", e); continue
    print("\n", os.path.relpath(t, root), "| rows:", len(df))
    print("  columns:", list(df.columns))
    v = [c for c in df.columns if "vendor" in str(c).lower()]
    k = [c for c in df.columns if any(w in str(c).lower() for w in ("code", "subject", "patient", "id"))]
    if v and k and meta is None:
        # prefer a single-letter vendor column (A/B/C/D) over a manufacturer-name column
        v.sort(key=lambda c: df[c].astype(str).str.len().mean())
        meta, vendor_col, code_col = df, v[0], k[0]

if meta is not None:
    print(f"\nUsing vendor column '{vendor_col}' and patient column '{code_col}'")
    print("Patients per vendor:")
    print(meta[vendor_col].value_counts().sort_index().to_string())
    print("Total patients in table:", len(meta))
    display(meta.head()) if "display" in dir() else print(meta.head())
elif tables:
    print("\nA table exists but no vendor column was recognised. Look at the column names above.")

In [ ]:
# 4. Labels: how many patients from each vendor have a ground-truth file?
import re

def is_label(path):
    n = os.path.basename(path).lower()
    return "_gt" in n or "label" in n or "mask" in n or "seg" in n

images = [f for f in files if ext_of(f) in (".nii.gz", ".nii", ".npy", ".npz", ".png", ".jpg", ".h5")]
labels = [f for f in images if is_label(f)]
print(f"{len(images)} image-type files, of which {len(labels)} look like label files")

if meta is not None:
    codes = [str(c) for c in meta[code_col]]
    has_img = {c: False for c in codes}
    has_lab = {c: False for c in codes}
    where = {c: "" for c in codes}
    for f in images:
        rel = os.path.relpath(f, root)
        for c in codes:
            if c in rel:
                if is_label(f): has_lab[c] = True
                else: has_img[c] = True
                parts = rel.split(os.sep)
                idx = next(i for i, p in enumerate(parts) if c in p)
                where[c] = "/".join(parts[:idx]) or "(top level)"
                break
    meta["_has_image"] = [has_img[c] for c in codes]
    meta["_has_label"] = [has_lab[c] for c in codes]
    meta["_folder"] = [where[c] for c in codes]
    summary = meta.groupby(vendor_col).agg(patients=(code_col, "count"),
                                           with_image=("_has_image", "sum"),
                                           with_label=("_has_label", "sum"))
    print("\nPer vendor:")
    print(summary.to_string())
    print("\nLabelled patients by folder and vendor:")
    print(pd.crosstab(meta["_folder"], meta[vendor_col], values=meta["_has_label"], aggfunc="sum").fillna(0).astype(int).to_string())
    missing = summary[summary["with_label"] == 0]
    if len(missing):
        print("\nWARNING: no labels found for vendor(s):", list(missing.index))
else:
    print("No usable metadata, so labels cannot be counted per vendor.")

In [ ]:
# 5. One labelled case in detail: shape, pixel spacing, label values, labelled time frames
import numpy as np

nii_labels = [f for f in labels if ext_of(f) in (".nii.gz", ".nii")]
if not nii_labels:
    print("No NIfTI label files. The copy has probably been converted to another format (see cell 2).")
    print("If the files are PNG or NPY slices, the pixel spacing is likely lost. Tell Claude what cell 2 printed.")
else:
    import nibabel as nib
    lab_path = sorted(nii_labels)[0]
    img_path = re.sub(r"_gt", "", lab_path)
    lab = nib.load(lab_path)
    print("Label file:", os.path.relpath(lab_path, root))
    print("  shape:", lab.shape, "| voxel size (mm and time):", tuple(round(float(z), 3) for z in lab.header.get_zooms()))
    data = np.asanyarray(lab.dataobj)
    print("  label values:", np.unique(data).tolist(), " (expected 0 background, 1 LV, 2 MYO, 3 RV)")
    if data.ndim == 4:
        frames = [t for t in range(data.shape[3]) if data[..., t].any()]
        print("  time frames that contain labels:", frames, " (expected two: ED and ES)")
        if meta is not None:
            row = meta[[str(c) in lab_path for c in meta[code_col]]]
            cols = [c for c in meta.columns if str(c).strip().upper() in ("ED", "ES")]
            if len(row) and cols:
                print("  metadata says:", {c: int(row.iloc[0][c]) for c in cols})
    if os.path.exists(img_path):
        img = nib.load(img_path)
        print("Image file:", os.path.relpath(img_path, root))
        print("  shape:", img.shape, "| matches label shape:", img.shape == lab.shape)

## What to look for

- **Cell 2:** `.nii.gz` files mean the copy is in the original format. Only `.png` or `.npy` files mean someone preprocessed it, and the pixel spacing is probably gone.
- **Cell 3:** there must be a table with a vendor column. The public release of M&Ms has 345 scans in total.
- **Cell 4:** every vendor needs a non-zero `with_label` count. The study trains on vendor B and tests on A, C and D, so all four matter.
- **Cell 5:** labels should be 0 to 3, and two time frames should carry labels.

Copy the printed output of cells 2 to 5 back to Claude.